### 1. Input

In [1]:
# ! pip install SpeechRecognition pydub
# ! pip install langdetect
# ! pip install openai-whisper
# ! pip install torch
# ! pip install pyaudio
# ! pip install ollama
# ! pip install gtts
# ! pip install pygame
# ! pip install playsound==1.2.2
# ! pip install edge_tts
# !pip install deep-translator
# !pip install easyocr
# !pip install transformers
import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect
import re
import ollama
from pydub import AudioSegment
import os
from gtts import gTTS
import pygame 
from playsound import playsound
import edge_tts
import asyncio
import json
import tkinter as tk
from tkinter import filedialog

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


#### 1.1. S2T

In [2]:
def audio_a_wav(audio):
    """ 
    Esta función devuelve un archivo de audio en formato .wav.
    Argurmento:
    - audio: archivo de audio en cualquier formato.
    """
    nombre, extension = os.path.splitext(audio)
    extension = extension.lower()

    if extension == ".wav":
        return audio
    else: 
        print("Convirtiendo el archivo a .wav.")
        archivo_nuevo = nombre + "convertido.wav"
        try:
            audio = AudioSegment.from_file(audio)
            audio.export(archivo_nuevo, format="wav")
            print("Archivo convertido a .wav.")
            return archivo_nuevo
        except:
            print("Error al convertir el archivo a .wav.")
            return None

In [3]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Modelo base
model = w.load_model("base", device=device)

def input_speech(audio = None):
    """ 
    Esta función transcribe un audio en formato .wav a texto o activa un micrófono en directo para guardar temporalmente el audio. Después de obtener el audio, lo transcribe y devuelve la transcripción y el idioma.
    Argumento:
    - audio: archivo de audio en formatio .wav (por defecto, audio = None).
    """
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        audio_a_wav(audio)
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente)
        result = model.transcribe(audio)
        # Se devuelve la transcripción del audio y el idioma
        return result["text"].strip(), result["language"]
    
    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio y eliminar esa parte
            r.adjust_for_ambient_noise(source, duration=1)
            r.pause_threshold = 2.0
            # Se escucha el audio
            audio_data = r.listen(source, phrase_time_limit=20)
            
            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())
                
            result = model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma
            return result["text"].strip(), result["language"]

#### 1.2. T2T

In [4]:
def input_text():
    """ 
    Esta función pide un texto al usuario, detectando el idioma. Devuelve el texto escrito y el idioma.
    """
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma
    try:
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"
    
    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

#### 1.3. Cleaning text

In [5]:
def cleaning_text(texto, idioma):
    """ 
    Esta función limpia un texto, para mejorar la legibilidad para después pasarselo a la IA. Devuelve el texto corregido.
    Argumentos:
    - texto: texto a limpiar.
    - idioma: idioma en el que está el texto.
    """
    # strip para eliminar espacios en blanco
    # capitalize para poner la primera letra en mayúscula
    texto_limpio = texto.strip().capitalize()

    # Muletillas a eliminar (español e inglés)
    muletillas = {
        "es": [r"\btipo+\b", r"\beh+\b", r"\besto+\b", r"\bpues+\b", r"\ben\s+plan\b"],
        "en": [r"\bum+\b", r"\beh+\b", r"\blike\b", r"\byou know\b"]
    }
    
    # Eliminamos las muletillas
    if idioma in muletillas:
        for i in muletillas[idioma]:
            texto_limpio = re.sub(i, "", texto_limpio, flags = re.IGNORECASE)

    # Eliminamos los n espacios que se han creado al eliminar las muletillas y lo sustitumos por un espacio
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

    return texto_limpio

### 2. T2T

In [6]:
def respuesta_texto(texto_limpio, idioma, opciones):
    """ 
    Esta función devuelve una respuesta por la IA (ollama) sobre x tema dado por el usuario.
    Argumentos:
    - texto_limpio: texto dado por el usuario y pasado por cleaning_text.
    - idioma: idioma detectado.
    - opciones: 

    """
    # Interfaz
    print(f"\n{opciones['titulo']}")
    print(opciones['opcion_1'])
    print(opciones['opcion_2'])
    print(opciones['opcion_3'])
    
    op = input(f"\n{opciones['pregunta']} ")

    estilo_instruccion = opciones['instrucciones_ia'].get(op, opciones['instrucciones_ia']["1"])

    # Qué queremos que nos devuelva la IA
    prompt = f"""
    Eres un guía turístico experto. 
    Tu objetivo es hablar y contar la historia sobre: {texto_limpio}
    Instrucción específica: {estilo_instruccion}
    IMPORTANTE: Debes responder OBLIGATORIAMENTE en el idioma correspondiente al código ISO: {idioma}.
    """
    
    # Llamamos a ollama
    respuesta_ia = ollama.chat(
        model="llama3", 
        messages=[{'role': 'user', 'content': prompt}]
    )
    
    # Devolvemos la respuesta generada
    return respuesta_ia['message']['content']

In [7]:
# PRUEBA
# respuesta_texto("información sobre la sagrada familia", "es")

### 3. Leer y traducir texto de una imagen

In [ ]:
from deep_translator import GoogleTranslator
import easyocr
import cv2
#from google.colab.patches import cv2_imshow

def leer_imagen(ruta_imagen, idioma=['en']):
  # Cargar imagen
  image = cv2.imread(ruta_imagen)

  # Cargar lector
  reader = easyocr.Reader(idioma)

  # Leer texto
  lectura = reader.readtext(image, paragraph=True, detail=0)
  return lectura


def cargar_imagen_tk():
    # Ventana, escondida
    root = tk.Tk()
    root.withdraw() 

    # Para que sea más cómodo lo dejo donde tengo las imágenes
    carpeta_inicial = "imagenes"

    ruta = filedialog.askopenfilename(
        initialdir=carpeta_inicial,
        title="Selecciona una imagen",
        filetypes=[
            ("PNG files", "*.png"),
            ("JPEG files", "*.jpg"),
            ("JPEG files", "*.jpeg"),
            ("BMP files", "*.bmp")
        ]
    )
    return ruta


def menu_idiomas():
  ruta = cargar_imagen_tk()
  print("""¿Qué alfabeto utiliza el texto a traducir?
  1. Latino
  2. Cirílico
  3. Chino tradicional
  4. Chino simplificado
  5. Japonés
  6. Coreano
  7. Salir
  """, flush=True)

  opcion = "aaaa"
  while opcion not in ["1","2","3","4","5","6","7"]:
    opcion = input("\nSelecciona una opción (1, 2, 3, 4, 5, 6 o 7): ")
    if opcion == "1":
      texto = leer_imagen(ruta, ['en'])
    if opcion == "2":
      texto = leer_imagen(ruta, ['ru','en'])
    if opcion == "3":
      texto = leer_imagen(ruta, ['ch_tra','en'])
    if opcion == "4":
      texto = leer_imagen(ruta, ['ch_sim', 'en'])
    if opcion == "5":
      texto = leer_imagen(ruta, ['ja', 'en'])
    if opcion == "6":
      texto = leer_imagen(ruta, ['ko','en'])
    if opcion == "7":
      return "¡Adiós!"
  
  texto_junto = " \n".join(texto)
  # Limpiar el texto a traducir
  texto_junto = re.sub('[$><|+`~]', '', texto_junto)
  texto_traducido = GoogleTranslator(source='auto', target='es').translate(texto_junto)
  
  return texto_traducido

In [15]:
t = menu_idiomas()
print(t)

¿Qué alfabeto utiliza el texto a traducir?
  1. Latino
  2. Cirílico
  3. Chino tradicional
  4. Chino simplificado
  5. Japonés
  6. Coreano
  7. Salir
  


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Mi nombre familiar es John: tengo 25 años y vivo en una casa pequeña con mi familia: tengo una madre, un padre y dos hermanos. El nombre de mi madre es Nary: y el nombre de mi padre es Tom. Los hermanos Ny son Jack y Mike. Jack tiene 18 años y Mike tiene 12 años. 
Tenemos un perro como mascota llamado Max Nax, es muy amigable: nos gusta ir al parque los fines de semana: amo mucho a mi familia.


### 4. Identificación de un monumento con una foto

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os.path
import torch
import requests

In [ ]:
import pandas as pd


def calcular_embeddings_monumentos():
    """ 
    Leer todos los nombres de los lugares de valor de la lista de patrimonio mundial de
    la unesco, se guardan los embeddings de estos lugares para poder identificarlos.
    Los nombres se obtienen de la base de datos disponible en: 
    https://data.unesco.org/explore/dataset/whc001/information/?flg=es-es&sort=date_inscribed
    """
    datos_patrimonio = pd.read_csv("datos/whc001.csv")
    
    titulos = ["a photo of" + nombre for nombre in datos_patrimonio['Name EN'].tolist()]
    
    # Cargar modelo
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    caracteristicas = {"titulos": [],
                       "caracteristicas": []}
    # Procesar el texto poco a poco, si no es muy exigente con la RAM
    for texto in titulos:
        text_inputs = processor(text=texto, return_tensors="pt", padding=True)
        with torch.no_grad(): # que no guarde gradientes (menos ram)
            text_features = model.get_text_features(**text_inputs)
        caracteristicas["titulos"].append(texto)
        caracteristicas["caracteristicas"].append(text_features)

    # Guardamos las características
    torch.save(caracteristicas, "datos/preproc_monumentos.pt")

# def identificar_monumento():
#     # Si no tenemos ya el archivo con las características de 
#     # los monumentos, calcularlos
#     if not os.path.isfile("datos/preproc_monumentos.pt"):
#         calcular_embeddings_monumentos()
    
#     model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
#     processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#     model.eval()

#     datos = torch.load("datos/preproc_monumentos.pt", weights_only=False)

#     titulos = datos["titulos"]
#     text_embeddings = datos["caracteristicas"]

#     image = Image.open("imagenes/prueba_monumento.jpeg")

#     inputs = processor(images=image, return_tensors="pt")

#     with torch.no_grad():
#         image_features = model.get_image_features(**inputs)
    
#     image_features = image_features / image_features.norm(dim=-1, keepdim=True)
#     text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

#     similarity = image_features @ text_embeddings.T  # producto escalar

#     best_idx = similarity.argmax().item()

#     print("Mejor match:", titulos[best_idx])

In [23]:
calcular_embeddings_monumentos()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
identificar_monumento()
# carac_texto = torch.load("datos/preproc_monumentos.pt")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'

### 4. Output (2. T2T o T2S)

In [8]:
#### VOZ MUY ROBOTIZADA
# def respuesta_audio(respuesta_ia, idioma):
#     voz = gTTS(text = respuesta_ia, lang = idioma)

#     voz_save = "respuesta_audio.mp3"
#     voz.save(voz_save)

#     pygame.mixer.init()
#     pygame.mixer.music.load(voz_save)
#     pygame.mixer.music.play()

#     while pygame.mixer.music.get_busy():
#         continue
    

In [9]:
async def respuesta_audio(respuesta_ia, idioma):
    """ 
    Esta función devuelve una respuesta por voz en un idioma dado (la respuesta es la que ha dado la IA, la pasamos a audio).
    Argumentos:
    - respuesta_ia: texto que ha dado la IA.
    - idioma: idioma en qué queremos la voz.
    """
    voces = await edge_tts.list_voices()
    voz_encontrada = None

    # Por defecto escogemos que el idioma "es" es "es-ES"
    if idioma == "es":
        for voz in voces:
            if "es-ES" in voz["ShortName"]:
                voz_encontrada = voz["ShortName"]
                break
    
    # Encontramos el idioma en el que queremos la respuesta
    if not voz_encontrada:
        for v in voces:
            if v["ShortName"].startswith(idioma):
                voz_encontrada = v["ShortName"]
                break

    # Si no encuentra la voz sobre ese idioma, por defecto usamos este
    if not voz_encontrada:
        voz_encontrada = "en-US-GuyNeural"
        
    # Guardamos temporalmente el audio
    archivo = "respuesta_voz.mp3"
    communicate = edge_tts.Communicate(respuesta_ia, voz_encontrada)
    await communicate.save(archivo)
    
    os.system(f"afplay {archivo}")
    os.remove(archivo)

In [10]:
# PRUEBA
# texto_ia = respuesta_texto("información sobre la sagrada familia", "es")
# await respuesta_audio(texto_ia, "es-ES")

In [11]:
# def menu_principal():
#     print("\n--- Guía turística---")
#     print("1. Escribir")
#     print("2. Hablar/Audio")
#     print("3. Hacer foto")
#     print("4. Salir")
    
#     opcion = input("\nSelecciona una opción (1, 2, 3 o 4): ")

#     texto_sucio, idioma = None, None
#     if opcion == "1":
#         texto_sucio, idioma = input_text()
    
#     elif opcion == "2":
#         print("\n¿Quieres (a) hablar o (b) pasar un archivo .wav?")
#         op = input("Selecciona una opción (a o b): ").lower()
#         if op == "a":
#             print("Escuchando...")
#             texto_sucio, idioma = input_speech()
#         else:
#             ruta = input("Introduce la ruta .wav: ")
#             texto_sucio, idioma = input_speech(audio=ruta)
            
#     elif opcion == "3":
#         return
        
#     elif opcion == "4":
#         print("...")
#         return
#     else:
#         print("Opción no válida.")
#         return

#     if texto_sucio and idioma:
#         texto_limpio = cleaning_text(texto_sucio, idioma)
#         print(f"\n Procesando texto: {texto_limpio}")

#         respuesta_final = respuesta_texto(texto_limpio, idioma)
        
#         print("\n Respuesta:")
#         print(respuesta_final)        
#         return respuesta_final

In [12]:
async def menu_principal():
    """ 
    Función para determinar en qué idioma quiere que esté la app el usuario (se lo pasamos a la IA para que lo traduzca).
    Se puede decir el idioma por voz o escribiéndolo.
    Se devuelve el código ISO del idioma.
    """
    # Qué idioma quiere el usuario?
    print("Configuración del idioma")
    print("Escribe el idioma (ej: 'español', 'english', 'italiano')")
    print("o pulsa ENTER para decir el idioma con la voz.")
    
    op_idioma = input("\n")

    if op_idioma == "":
        print("Escuchando...")
        # Usamos Whisper para detectar qué idioma quiere el usuario
        texto, cod_idioma = input_speech() 
        print(f"Idioma detectado: {texto} ({cod_idioma})")
    else:
        # También puede escribir el idioma que quiere
        prompt_iso = f"Dame solo el código ISO 639-1 (2 letras) para el idioma: {op_idioma}. Ejemplo: es, en, fr. No escribas nada más."
        res = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt_iso}])
        cod_idioma = res['message']['content'].strip().lower()
        print(f"Idioma detectado: {op_idioma} ({cod_idioma})")

    # Devolvemos el código del idioma
    return cod_idioma

In [13]:
# Menú a traducir según el idioma deseado por el usuario
MENU_COMPLETO = {
    "menu": {
        "bienvenida": "Guía turística IA",
        "opciones": [
            "1. Escribir una pregunta (Texto)",
            "2. Hablar con el guía o subir un audio (Audio)",
            "3. Hacer foto a un monumento (Cámara)",
            "4. Salir"
        ],
        "instruccion": "Selecciona una opción (1, 2, 3 o 4):"
    },
    "mensajes": {
        "pregunta_texto": "Escribe tu duda:",
        "pregunta_audio_modo": "¿Qoperes (a) hablar o (b) pasar un archivo .wav?",
        "instruccion_audio": "Selecciona una opción (a o b):",
        "escuchando": "Escuchando...",
        "ruta_archivo": "Introduce la ruta del archivo .wav:",
        "procesando": "Procesando información.",
        "foto": "Haz una foto.",
        "error_opcion": "Opción no válida. Inténtelo de nuevo.",
        "despedida": "¡Adiós!"
    },
    "estilos": {
        "titulo": "--- Opciones de estilo ---",
        "opcion_1": "1. Resumen breve",
        "opcion_2": "2. Historia detallada",
        "opcion_3": "3. Datos curiosos",
        "pregunta": "Selecciona el tipo de información deseada (1, 2 o 3):",
        "instrucciones_ia": {
            "1": "Hazme un resumen breve.",
            "2": "Hazme una explicación detallada y técnica.",
            "3": "Cuéntame curiosidades."
        }
    }
}


async def menu_traducido(cod_idioma):
    """ 
    Función para traducir MENU_COMPLETO al idioma deseado.
    Argumento:
    - cod_idioma: código del idioma al que se va a traducir toda la app.
    """
    print(f"Traducción al idioma: [{cod_idioma}]...")
    
    # prompt para ollama apra traducir MENU_COMPLETO
    prompt = f"""
    Traduce este JSON al idioma '{cod_idioma}'.
    Mantén las claves originales y traduce solo los valores.
    Responde SOLO el JSON.
    JSON: {json.dumps(MENU_COMPLETO, ensure_ascii=False)}
    """
    
    # Llamamos a ollama
    response = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
    res_text = response['message']['content'].strip()
    
    res_text = res_text.replace("```json", "").replace("```", "").strip()
    
    try:
        return json.loads(res_text)
    except:
        return MENU_COMPLETO 
    

async def app():
    """ 
    Función ("app") que une todas la funciones.
    """
    # Obtenemos el idioma deseado
    cod_idioma = await menu_principal() 
    
    # Traducimos la interfaz del menú
    interfaz = await menu_traducido(cod_idioma)
    
    while True:
        print(f"\n--- {interfaz['menu']['bienvenida']} ---")
        for opt in interfaz['menu']['opciones']:
            print(opt)
        
        opcion = input(f"\n{interfaz['menu']['instruccion']} ")
        texto_sucio, idioma_final = None, cod_idioma

        # Opción para escribir
        if opcion == "1":
            texto_sucio = input(f"{interfaz['mensajes']['pregunta_texto']} ")
        
        # Opción para hablar
        elif opcion == "2":
            print(f"\n{interfaz['mensajes']['pregunta_audio_modo']}")
            sub_opcion = input(f"{interfaz['mensajes']['instruccion_audio']} ").lower()
            
            # Hablando
            if sub_opcion == "a":
                print(interfaz['mensajes']['escuchando'])
                texto_sucio, idioma_final = input_speech() 
            # Subiendo un audio
            else:
                ruta = input(f"{interfaz['mensajes']['ruta_archivo']} ")
                texto_sucio, idioma_final = input_speech(audio=ruta)

        # Hacer foto (FALTA AÑADIR)
        elif opcion == "3":
            print(interfaz['mensajes']['foto'])
            continue

        # Salida
        elif opcion == "4":
            print(interfaz['mensajes']['despedida'])
            break
        
        # Error
        else:
            print(interfaz['mensajes']['error_opcion'])
            continue

        if texto_sucio:
            print(f"\n{interfaz['mensajes']['procesando']}")
            texto_limpio = cleaning_text(texto_sucio, idioma_final)
            respuesta = respuesta_texto(texto_limpio, idioma_final, interfaz['estilos'])
            
            print(f"\nRespuesta:\n{respuesta}")

In [ ]:
#await app() 

# Me acaba de decir que la ciudad de las artes y las ciencias está en barcelna
# Respuesta:
# ¡Claro! Me alegra poder hablar sobre la ciudad de las artes y las ciencias, ubicada en Barcelona, España. A continuación, te doy un resumen breve:

# La Ciudad de las Artes y las Ciencias (en catalán, Ciutat de les Arts i les Ciències) es un complejo arquitectónico y cultural situado en el distrito de l'Eixample, en el corazón de Barcelona. Fue diseñada por los arquitectos Félix Candela, Rafael Moneo y Santiago Calatrava, y se inauguró en 1990.

# El complejo consta de varios edificios y estructuras que reflejan la fusión de arte, ciencia y tecnología. Entre ellos destacan el Palau de les Arts Reina Sofía (teatro y centro de música), el Museo de las Ciencias Príncipe Felipe, la Torre l'Ascensor, el Hemisfèric (un planetario y observatorio) y la Ciudad de las Esculturas.

# La ciudad es un lugar perfecto para la contemplación, la educación y el entretenimiento. Ofrece visitas guiadas, exhibiciones temporales y permanentes, conciertos y espectáculos, lo que la convierte en uno de los atractivos turísticos más populares de Barcelona.

# ¡Espero que disfrutes visitando esta maravillosa ciudad!

Configuración del idioma
Escribe el idioma (ej: 'español', 'english', 'italiano')
o pulsa ENTER para decir el idioma con la voz.
Idioma detectado: español (es)
Traducción al idioma: [es]...

--- Guía turística Inteligente ---
1. Escribir una pregunta (Texto)
2. Hablar con el guía o subir un audio (Audio)
3. Hacer foto a un monumento (Cámara)
4. Salir

Procesando información.

--- Opciones de estilo ---
1. Resumen breve
2. Historia detallada
3. Datos curiosos

Respuesta:
¡Claro! Me alegra poder hablar sobre la ciudad de las artes y las ciencias, ubicada en Barcelona, España. A continuación, te doy un resumen breve:

La Ciudad de las Artes y las Ciencias (en catalán, Ciutat de les Arts i les Ciències) es un complejo arquitectónico y cultural situado en el distrito de l'Eixample, en el corazón de Barcelona. Fue diseñada por los arquitectos Félix Candela, Rafael Moneo y Santiago Calatrava, y se inauguró en 1990.

El complejo consta de varios edificios y estructuras que reflejan la fusión d